# Intraday Entry/Exit Strategy (TensorFlow)
This notebook provides a TensorFlow implementation of a dueling DQN LSTM policy (entry/exit) adapted to the `SuperTrader.AI` codebase.

Notes: the repository uses a lookback=30 and base features=32 (see `models/dqn_network.py`). This notebook runs a short smoke-test and demonstrates the policy + GPU setup.

## 1) Install & GPU setup
Install dependencies if missing and configure TensorFlow to use GPU. In hosted runtimes (Colab) switch runtime type to GPU. Locally, ensure CUDA/cuDNN and matching TensorFlow build are installed.

In [ ]:
# Install common packages (uncomment to run)
# !pip install --upgrade pip
# !pip install tensorflow pandas numpy matplotlib scikit-learn joblib

import os
import tensorflow as tf
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('intraday-notebook')

# Print TF + GPU info
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs found:', gpus)
if gpus:
    try:
        # Enable memory growth for GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(len(gpus), 'Physical GPUs,', len(logical_gpus), 'Logical GPUs')
    except Exception as e:
        print('Could not set memory growth:', e)
else:
    print('No GPU available - the code will still run on CPU.')

## 2) Add codebase to PYTHONPATH and import models
Make the repo importable so we can reference `models.dqn_network` and the environment.

In [ ]:
import sys
from pathlib import Path
repo_root = Path('/Users/muaazshaikh/SuperTrader.AI').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Added to PYTHONPATH:', repo_root)

# Try importing repository modules with graceful fallback
try:
    import models.dqn_network as dqn_network
    import models.intraday_environment as intraday_environment
    import models.training as training_module
    print('Imported models modules: dqn_network, intraday_environment, training')
except Exception as e:
    print('Import failed (okay for demo). Error:', e)
    dqn_network = None
    intraday_environment = None
    training_module = None

# For iterative dev in notebook: reload modules after editing
from importlib import reload
if dqn_network is not None:
    reload(dqn_network)
if intraday_environment is not None:
    reload(intraday_environment)
if training_module is not None:
    reload(training_module)

## 3) Inspect model source files (quick programmatic view)
Print key model files so you can confirm class names and adapt calls.

In [ ]:
files = [
    repo_root / 'models' / '__init__.py',
    repo_root / 'models' / 'dqn_network.py',
    repo_root / 'models' / 'intraday_environment.py',
    repo_root / 'models' / 'training.py'
]
for f in files:
    print('---', f.name, '---')
    try:
        print(f.read_text()[:4000])  # print first chunk
    except Exception as e:
        print('Could not read', f, 'Error:', e)

## 4) Prepare dataset (placeholder + example feature engineering)
You should replace the CSV path with your actual intraday price data. Example shows common intraday features and scalers.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Example: user provides path_to_csv. If not provided, we generate synthetic data.
path_to_csv = None  # e.g. '/path/to/nifty_1min.csv'
if path_to_csv is not None and Path(path_to_csv).exists():
    df = pd.read_csv(path_to_csv, parse_dates=['timestamp'], index_col='timestamp')
    print('Loaded CSV with', len(df), 'rows')
else:
    # Generate a short synthetic intraday day of 6 hours * 60 = 360 bars (1-min)
    np.random.seed(42)
    timestamps = pd.date_range('2025-01-01 09:15', periods=360, freq='1min')
    base = 22000 + np.cumsum(np.random.randn(len(timestamps)) * 2)
    df = pd.DataFrame({
        'open': base + np.random.randn(len(timestamps)),
        'high': base + np.abs(np.random.randn(len(timestamps)) * 2),
        'low': base - np.abs(np.random.randn(len(timestamps)) * 2),
        'close': base,
        'volume': np.random.randint(100, 5000, len(timestamps))
    }, index=timestamps)
    print('Created synthetic data with', len(df), 'bars')

# Feature engineering: returns, rolling mean, volatility
df['ret_1'] = df['close'].pct_change().fillna(0)
df['ret_5'] = df['close'].pct_change(5).fillna(0)
df['rsi_14'] = (100 - (100 / (1 + df['ret_1'].rolling(14).mean().fillna(0))))
df['sma_20'] = df['close'].rolling(20).mean().fillna(method='bfill')
df['vol_ewma'] = df['ret_1'].ewm(span=30).std().fillna(0.001)

feature_cols = ['open','high','low','close','volume','ret_1','ret_5','rsi_14','sma_20','vol_ewma']
scaler = StandardScaler()
X = scaler.fit_transform(df[feature_cols])
print('Feature matrix shape:', X.shape)

## 5) Instantiate IntradayEnvironment (example)
The repository environment class may be named `IntradayTradingEnv` (or similar). We'll attempt to instantiate it if available, otherwise we use a tiny wrapper to simulate step/reset semantics.

In [ ]:
# Try to instantiate environment from repository if present
EnvClass = None
try:
    # Many repo variants use IntradayTradingEnv or IntradayMarket names
    if intraday_environment is not None:
        EnvClass = getattr(intraday_environment, 'IntradayTradingEnv', None) or getattr(intraday_environment, 'IntradayMarket', None)
    if EnvClass is not None:
        env = EnvClass(data=df, initial_capital=500000, lot_size=50, max_lots=3)
        print('Instantiated repo environment:', EnvClass)
    else:
        raise ImportError('Env class not found in models.intraday_environment')
except Exception as e:
    print('Could not instantiate repo environment (falling back to demo wrapper). Error:', e)
    # Demo fallback environment with minimal API: reset(), step(action)
    class DemoEnv:
        def __init__(self, df, window=30):
            self.df = df
            self.window = window
            self.step_idx = 0
            self.position = 0
            self.entry_price = 0.0
        def reset(self):
            self.step_idx = self.window
            self.position = 0
            self.entry_price = 0.0
            return self._get_state()
        def _get_state(self):
            window = self.df.iloc[self.step_idx-self.window:self.step_idx]
            return window[feature_cols].values.astype(np.float32)
        def step(self, action):
            # action: 0=short,1=hold,2=long (mapping chosen to match repo)
            price = float(self.df['close'].iloc[self.step_idx])
            reward = 0.0
            done = False
            info = {}
            # simplified P&L: if position != 0, unrealized pnl = (price - entry)*position
            if action == 2 and self.position == 0:  # enter long
                self.position = 1
                self.entry_price = price
            elif action == 0 and self.position == 0:  # enter short
                self.position = -1
                self.entry_price = price
            elif action == 1 and self.position != 0:  # hold action treated as exit here for demo
                # exit
                reward = (price - self.entry_price) * self.position
                self.position = 0
                self.entry_price = 0.0
            # move forward
            self.step_idx += 1
            if self.step_idx >= len(self.df):
                done = True
            next_state = self._get_state() if not done else np.zeros((self.window, len(feature_cols)), dtype=np.float32)
            return next_state, float(reward), done, info
    env = DemoEnv(df)
    print('Using DemoEnv for the smoke-run')

## 6) Build / load DQN networks (policy & target)
We'll implement a TensorFlow Keras dueling LSTM matching the repo's shape: lookback=30, features ~ len(feature_cols) (demo).

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers

class DuelingDQN_LSTM(Model):
    def __init__(self, n_features, n_actions=3, lstm1_units=64, lstm2_units=32, dropout=0.1):
        super().__init__()
        self.lstm1 = layers.LSTM(lstm1_units, return_sequences=True)
        self.lstm2 = layers.LSTM(lstm2_units, return_sequences=False)
        self.drop = layers.Dropout(dropout)
        self.val_fc = layers.Dense(lstm2_units, activation='relu')
        self.val_out = layers.Dense(1)
        self.adv_fc = layers.Dense(lstm2_units, activation='relu')
        self.adv_out = layers.Dense(n_actions)
    def call(self, x, training=False):
        x = self.lstm1(x, training=training)
        x = self.lstm2(x, training=training)
        x = self.drop(x, training=training)
        V = self.val_out(self.val_fc(x))
        A = self.adv_out(self.adv_fc(x))
        A_mean = tf.reduce_mean(A, axis=1, keepdims=True)
        Q = V + (A - A_mean)
        return Q

# Instantiate policy & target models for demo
n_features = len(feature_cols)
policy_model = DuelingDQN_LSTM(n_features=n_features, n_actions=3)
target_model = DuelingDQN_LSTM(n_features=n_features, n_actions=3)
# Build models by calling once
dummy = tf.zeros((1, 30, n_features))
policy_model(dummy)
target_model(dummy)
# Copy weights to target
target_model.set_weights(policy_model.get_weights())
optimizer = optimizers.Adam(learning_rate=1e-4)
print('Policy & target models created. Policy summary:')
policy_model.summary()

## 7) Entry & exit policy (epsilon-greedy + mapping)
Action mapping used in the repo: 0=SHORT, 1=HOLD, 2=LONG. We'll implement epsilon-greedy and enforce the time-safety rule: if minutes_to_close < 15 => HOLD.

In [ ]:
import numpy as np

class TFTradingAgent:
    def __init__(self, policy_model, target_model, n_actions=3, gamma=0.3, epsilon=0.5, epsilon_min=0.01, epsilon_decay=0.995):
        self.policy_model = policy_model
        self.target_model = target_model
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.optimizer = optimizers.Adam(learning_rate=1e-4)

    def decide_action(self, state, mode='train', minutes_to_close=180.0):
        # Force HOLD close to market close (safety rule from repo)
        if minutes_to_close < 15:
            return 1, {'reason': 'time_constraint'}

        if mode == 'train' and np.random.rand() < self.epsilon:
            return np.random.randint(0, self.n_actions), {'mode': 'explore', 'epsilon': self.epsilon}

        q = self.policy_model(state[np.newaxis].astype(np.float32))
        q = q.numpy()[0]
        action = int(np.argmax(q))
        return action, {'mode': 'exploit', 'q_values': q.tolist()}

    def remember(self, *args, **kwargs):
        # For demo, memory handled externally by ReplayBuffer below
        pass

    def train_on_batch(self, states, actions, rewards, next_states, dones):
        # Compute targets using Bellman equation with target_model
        next_q = self.target_model(next_states).numpy()
        max_next_q = np.max(next_q, axis=1)
        targets = rewards + (1 - dones) * self.gamma * max_next_q

        with tf.GradientTape() as tape:
            q_values = self.policy_model(states)
            # gather predicted q for actions
            action_masks = tf.one_hot(actions, self.n_actions)
            pred_q = tf.reduce_sum(q_values * action_masks, axis=1)
            loss = tf.reduce_mean(tf.keras.losses.Huber()(targets, pred_q))
        grads = tape.gradient(loss, self.policy_model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.policy_model.trainable_variables))

        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

        return float(loss)

## 8) Replay buffer & training utilities

In [ ]:
import collections, random
class ReplayBuffer:
    def __init__(self, capacity=5000):
        self.buf = collections.deque(maxlen=capacity)
    def add(self, s,a,r,ns,d):
        self.buf.append((s,a,r,ns,d))
    def sample(self, batch_size):
        batch = random.sample(self.buf, min(batch_size, len(self.buf)))
        s,a,r,ns,d = map(np.array, zip(*batch))
        return s, a, r.astype(np.float32), ns, d.astype(np.float32)
    def __len__(self):
        return len(self.buf)

buffer = ReplayBuffer(capacity=2000)
agent = TFTradingAgent(policy_model, target_model)
print('ReplayBuffer & agent ready. Buffer size:', len(buffer))

## 9) Training loop (smoke run for 1-2 episodes)
We'll run a short loop to demonstrate end-to-end action selection and training steps. This is NOT full training; adapt the loops and logging from `models/training.py` for full training.

In [ ]:
# Simple smoke training loop over the demo environment
episodes = 2
batch_size = 16
target_update_freq = 50
for ep in range(episodes):
    state = env.reset()
    done = False
    total_reward = 0.0
    step = 0
    while not done and step < 200:
        # For demo, minutes_to_close reduces with step (simple proxy)
        minutes_to_close = max(0, 180 - step)
        action, info = agent.decide_action(state, mode='train', minutes_to_close=minutes_to_close)
        next_state, reward, done, env_info = env.step(action)
        buffer.add(state, action, reward, next_state, float(done))
        state = next_state
        total_reward += reward
        step += 1
        # Train if buffer has enough samples
        if len(buffer) >= batch_size:
            s_b, a_b, r_b, ns_b, d_b = buffer.sample(batch_size)
            loss = agent.train_on_batch(s_b, a_b, r_b, ns_b, d_b)
        # Periodically sync target network
        if step % target_update_freq == 0:
            target_model.set_weights(policy_model.get_weights())
    print(f'Episode {ep+1} finished. Steps: {step}, Total reward: {total_reward:.4f}, Epsilon: {agent.epsilon:.3f}')

## 10) Evaluate agent & simple backtest
Run the agent greedily over the same demo env to collect P&L and trades.

In [ ]:
# Greedy evaluation
state = env.reset()
done = False
equity = [0.0]
trade_log = []
step = 0
while not done and step < 500:
    action, info = agent.decide_action(state, mode='eval', minutes_to_close=180-step)
    next_state, reward, done, info = env.step(action)
    if reward != 0:
        trade_log.append({'step': step, 'reward': reward})
    equity.append(equity[-1] + reward)
    state = next_state
    step += 1
print('Evaluation complete. Trades:', len(trade_log))
import matplotlib.pyplot as plt
plt.plot(equity)
plt.title('Demo equity curve (synthetic demo)')
plt.xlabel('Step')
plt.ylabel('Cumulative reward')
plt.grid(True)
plt.show()

## 11) Save / load trained model weights
Save model weights and scaler for later inference.

In [ ]:
save_dir = repo_root / 'models'
save_dir.mkdir(exist_ok=True)
policy_path = save_dir / 'tf_policy_weights'
policy_model.save_weights(str(policy_path))
print('Saved policy weights to', policy_path)
# To load: policy_model.load_weights(str(policy_path))
import joblib
scaler_path = save_dir / 'feature_scaler.pkl'
joblib.dump(scaler, scaler_path)
print('Saved scaler to', scaler_path)

## 12) Visualize results (equity curve, trades)
We plotted a simple equity curve above. For real data, align entry/exit timestamps and mark trades on a price chart.

## 13) Quick smoke tests / assertions
A few checks to ensure shapes and GPU visibility.

In [ ]:
# Smoke assertions
s = env.reset()
assert s.shape == (30, len(feature_cols)), f'Unexpected state shape: {s.shape}'
q = policy_model(s[np.newaxis].astype(np.float32))
assert q.shape[-1] == 3, 'Policy output should have 3 actions'
print('Smoke checks passed. GPU available:', len(tf.config.list_physical_devices('GPU'))>0)

---
That completes the demo notebook for entry/exit policy in TensorFlow.
Next steps (suggested):
- Replace the `DemoEnv` with the repo environment (`IntradayTradingEnv`) and adapt the feature columns to match the state builder (lookback=30, feature_count=32 or 68).
- Extend the training loop per `models/training.py` to run over many episodes, add TensorBoard logging, and checkpointing.
- If you'd like, I can: (a) wire this notebook to the real env/data in the repo, or (b) convert the training loop into a script matching `training.py`. Which do you prefer?